Experiments for the LSTM Neural Network Model
---
Performs grid search on __ different combinations to determine the best NN model for dementia classifcation.

Grid search is run on all three transcript types to determine the best model for each PFT, CTD, and SFT.

In [3]:
import sys
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer

sys.path.append(str(Path("..").resolve()))

DATA_PATH = Path("..") / "data" / "transcripts_cleaned.csv"

transcript_df = pd.read_csv(DATA_PATH)

print(transcript_df.shape)
transcript_df.head()

(157, 6)


,Record-ID,Class,Transcript_PFT,Transcript_CTD,Transcript_SFT,Label
0,Process-rec-001,MCI,"people, partner, plate, platter, pants, porter...",NaN,"<pause_medium> giraffe, kangaroo, lion, tiger,...",1
1,Process-rec-002,MCI,"<pause_short> pipe, plane, people <pause_mediu...",<pause_medium> there’s a lad stood on the stoo...,"<pause_short> dogs, cats, birds <pause_short> ...",1
2,Process-rec-003,MCI,"um <pause_short> purple, pale, placid <pause_s...","<pause_medium> um, the picture is of a kitchen...","cow, bull, ewe, ram, chicken, goose, um <sigh>...",1
3,Process-rec-004,MCI,plank <pause_short> pool <pause_short> swimmin...,"a mother presumably, or a fe, an adult female ...",um <pause_short> impala <pause_short> er cheet...,1
4,Process-rec-005,MCI,"it’s er pillock, er post box, er pyracanthas, ...","‘50s style er scene of domestic um confusion, ...","dog, cat, giraffe, wallaby, kangaroo, tortoise...",1


In [4]:
TRANSCRIPT_COLS = ["Transcript_PFT", "Transcript_CTD", "Transcript_SFT"]

print("NaN counts per transcript type: ")
for col in TRANSCRIPT_COLS:
    n_nans = transcript_df[col].isna().sum()
    print(f"{col}: {n_nans} NaNs")

NaN counts per transcript type: 
Transcript_PFT: 0 NaNs
Transcript_CTD: 1 NaNs
Transcript_SFT: 5 NaNs


In [10]:
from transcript_preprocessing import load_transcript_splits
df_by_transcript = load_transcript_splits(DATA_PATH)

for col in TRANSCRIPT_COLS:
    print(f"After removing NaNs for {col}: {len(df_by_transcript[col])} samples")

for col, df_clean in df_by_transcript.items():
    print(f"\nLabel counts for {col}:")
    print(df_clean["Class"].value_counts())


After removing NaNs for Transcript_PFT: 157 samples
After removing NaNs for Transcript_CTD: 156 samples
After removing NaNs for Transcript_SFT: 152 samples

Label counts for Transcript_PFT:
Class
HC          82
MCI         59
Dementia    16
Name: count, dtype: int64

Label counts for Transcript_CTD:
Class
HC          82
MCI         58
Dementia    16
Name: count, dtype: int64

Label counts for Transcript_SFT:
Class
HC          78
MCI         58
Dementia    16
Name: count, dtype: int64


In [ ]:
# Analyze transcript lengths and vocab sizes to determine how to set max lengths anf if voacb limiting is necessary
length_stats = {}

for col in TRANSCRIPT_COLS:
    print(f"\n{col}:")
    
    df = df_by_transcript[col]
    texts = df[col].astype(str).tolist()
    
    # Fit tokenizer for analysis
    tokenizer = Tokenizer(oov_token="<UNK>")
    tokenizer.fit_on_texts(texts)
    
    vocab_size = len(tokenizer.word_index)
    
    # Convert to sequences
    sequences = tokenizer.texts_to_sequences(texts)
    lengths = np.array([len(seq) for seq in sequences])
    
    length_stats[col] = {
        "lengths": lengths,
        "tokenizer": tokenizer,
        "vocab_size": vocab_size,
    }
    
    print(f"Number of samples: {len(lengths)}")
    print(f"Vocab size (unique tokens): {vocab_size}")
    print(f"Min length:       {lengths.min()}")
    print(f"Max length:       {lengths.max()}")
    print(f"Mean length:      {lengths.mean():.2f}")



Transcript_PFT:
Number of samples: 157
Vocab size (unique tokens): 1267
Min length:       13
Max length:       94
Mean length:      47.58

Transcript_CTD:
Number of samples: 156
Vocab size (unique tokens): 1400
Min length:       13
Max length:       474
Mean length:      163.89

Transcript_SFT:
Number of samples: 152
Vocab size (unique tokens): 931
Min length:       25
Max length:       121
Mean length:      62.91


In [12]:
MAX_LEN_PFT = 94
MAX_LEN_CTD = 474
MAX_LEN_SFT = 121

MAX_LEN_BY_TRANSCRIPT = {
    "Transcript_PFT": MAX_LEN_PFT,
    "Transcript_CTD": MAX_LEN_CTD,
    "Transcript_SFT": MAX_LEN_SFT,
}

In [17]:
from transcript_preprocessing import get_stratified_kfold_splits
from lstm_nn.utils import prepare_fold_data

# Verify padding/tokenization for each transcript type (for first fold only)
for transcript_col in TRANSCRIPT_COLS:
    print(f"\n{transcript_col}:")
    
    df = df_by_transcript[transcript_col]
    max_len = MAX_LEN_BY_TRANSCRIPT[transcript_col]
    
    y = df["Label"].values
    
    # Run Stratified K-Fold, and inspect the first fold
    for fold_idx, train_idx, val_idx in get_stratified_kfold_splits(
        transcript_df=df,
        transcript_col=transcript_col,
        label_col="Label",
        n_splits=5,
        seed=42,
    ):
        print(f"Fold {fold_idx}")
        
        X_train, y_train, X_val, y_val, tokenizer, vocab_size = prepare_fold_data(
            df=df,
            transcript_col=transcript_col,
            label_col="Label",
            train_idx=train_idx,
            val_idx=val_idx,
            max_len=max_len,
        )
        
        print("  X_train shape:", X_train.shape)
        print("  X_val shape:  ", X_val.shape)
        print("  y_train shape:", y_train.shape)
        print("  y_val shape:  ", y_val.shape)
        print("  max_len used: ", max_len)
        print("  vocab_size:   ", vocab_size)
        break



Transcript_PFT:
Fold 1
  X_train shape: (125, 94)
  X_val shape:   (32, 94)
  y_train shape: (125,)
  y_val shape:   (32,)
  max_len used:  94
  vocab_size:    1125

Transcript_CTD:
Fold 1
  X_train shape: (124, 474)
  X_val shape:   (32, 474)
  y_train shape: (124,)
  y_val shape:   (32,)
  max_len used:  474
  vocab_size:    1298

Transcript_SFT:
Fold 1
  X_train shape: (121, 121)
  X_val shape:   (31, 121)
  y_train shape: (121,)
  y_val shape:   (31,)
  max_len used:  121
  vocab_size:    873
